In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

input_csv_path = "input.csv"

out_y_csv = "properties.csv"
out_x_csv = "feature_list.csv"

n_bits = 2048
radius = 2

df = pd.read_csv("../data/zinc_dft/zinc_dft.csv")

# remove negative values and NaN
df["absorption wavelength"] = pd.to_numeric(df["absorption wavelength"], errors="coerce")
df["intensity"] = pd.to_numeric(df["intensity"], errors="coerce")
df = df.dropna(subset=["smiles", "absorption wavelength", "intensity"])
df = df[(df["absorption wavelength"] >= 0) & (df["intensity"] >= 0)].copy()

fps = []
keep_rows = [] # to remove failed rows

mfgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

for i, row in df.iterrows():
    smi = str(row["smiles"])
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue

    fp = mfgen.GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    fps.append(arr)
    keep_rows.append(i)

df_ok = df.loc[keep_rows].copy().reset_index(drop=True)
X = np.vstack(fps).astype(np.int8)

y = df_ok[["absorption wavelength", "intensity"]]
y.to_csv(out_y_csv, index=False)

np.savetxt(out_x_csv, X, fmt="%d", delimiter=",") # too large to commit (350MB)

print(f"Rows kept: {len(df_ok)}")

Rows kept: 87628
